In [ ]:
from pathlib import Path
from safetensors import safe_open

# Local checkpoint path (already downloaded in this environment)
ckpt_path = Path("/home/wgz/.cache/huggingface/hub/models--lpiccinelli--unidepth-v2-vitl14/snapshots/52b349b514bd8b47642f67ac78cb7b5dc5c51dd9/model.safetensors")

if not ckpt_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

def count_params_safetensors(path: Path):
    total = 0
    per_prefix = {}
    with safe_open(str(path), framework="pt", device="cpu") as f:
        keys = list(f.keys())
        for k in keys:
            shape = f.get_tensor(k).shape
            n = 1
            for d in shape:
                n *= d
            total += n
            prefix = k.split(".")[0]
            per_prefix[prefix] = per_prefix.get(prefix, 0) + n
    return total, per_prefix

total, per_prefix = count_params_safetensors(ckpt_path)
print(f"Checkpoint: {ckpt_path}")
print(f"Total parameters: {total:,}")
print(f"Total parameters (M): {total/1e6:.2f}M")
print("\nTop-level parameter groups:")
for name, n in sorted(per_prefix.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name:20s} {n/1e6:10.2f}M")

Checkpoint: /home/wgz/.cache/huggingface/hub/models--lpiccinelli--unidepth-v2-vitl14/snapshots/52b349b514bd8b47642f67ac78cb7b5dc5c51dd9/model.safetensors
Total parameters: 353,831,043
Total parameters (M): 353.83M

Top-level parameter groups:
  pixel_encoder            304.37M
  pixel_decoder             49.46M


In [ ]:
# Optional cross-check by instantiating the model class (can fail if network/proxy blocks HF).
try:
    from unidepth.models import UniDepthV2
    import torch

    model = UniDepthV2.from_pretrained("lpiccinelli/unidepth-v2-vitl14")
    total_model = sum(p.numel() for p in model.parameters())
    trainable_model = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("Model object total params:", f"{total_model:,}")
    print("Model object trainable params:", f"{trainable_model:,}")
except Exception as e:
    print("Model instantiation check skipped/failed:", repr(e))

/home/wgz/UniDepth/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/wgz/UniDepth/.venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/wgz/UniDepth/unidepth/utils/chamfer_distance.py:9: UserWarning: !! To run evaluation you need KNN. Please compile KNN: `cd unidepth/ops/knn with && bash compile.sh`.
  warnings.warn(


Cannot import NystromAttention, you can not run original UniDepth. UniDepthV2 is available.


xFormers not available
xFormers not available


Not loading pretrained weights for backbone
EdgeGuidedLocalSSI reverts to a non cuda-optimized operation, you will experince large slowdown, please install it:  `cd ./unidepth/ops/extract_patches && bash compile.sh`
Model object total params: 353,831,043
Model object trainable params: 353,223,811
